## Plain Transformer (FiLM on attributes) — Kaggle-ready training

This notebook trains the **decoder-only baseline** Transformer (matched to the VAE decoder hyperparameters) conditioned only on per-bar attribute embeddings `A_k` via FiLM.

### What you need to do on Kaggle
- Attach your memmap datasets as Kaggle Datasets.
- Your layout is expected to be:
  - `/kaggle/input/datasets/<user>/<preprocessed-memmaptrain>/train/...`
  - `/kaggle/input/datasets/<user>/<preprocessed-memmapval>/val/...`
- Set `KAGGLE_USER`, `TRAIN_DATASET`, and `VAL_DATASET` below.
- Click **Run All**.

### Outputs
- Checkpoints and logs are written to `/kaggle/working/`.


In [ ]:
from __future__ import annotations

import os
import sys
import json
import time
from pathlib import Path

import numpy as np
import torch

print('cwd:', os.getcwd())
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('torch cuda:', torch.version.cuda)
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

INPUT_ROOT = Path('/kaggle/input')
WORKING_ROOT = Path('/kaggle/working')
print('input exists:', INPUT_ROOT.exists())
print('working exists:', WORKING_ROOT.exists())

# List datasets attached to the notebook
if INPUT_ROOT.exists():
    print('--- /kaggle/input ---')
    for p in sorted(INPUT_ROOT.iterdir()):
        if p.is_dir():
            print(' -', p.name)


In [ ]:
# -------------------------
# Config (edit these on Kaggle)
# -------------------------

# Your Kaggle layout is: /kaggle/input/datasets/<user>/<dataset-slug>/...
KAGGLE_USER = "karandeepshoker"
TRAIN_DATASET = "preprocessed-memmaptrain"
VAL_DATASET = "preprocessed-memmapval"
# TEST_DATASET = "preprocessed-memmaptest"  # optional

# Inside each dataset, where do the split folders live?
# Set to Path('.') if the dataset contains train/ or val/ at its root.
# Set to Path('preprocessed_memmap') if it contains preprocessed_memmap/train, etc.
SPLIT_ROOT_REL = Path('.')

MAX_STEPS = 10_000  # optimizer updates
BATCH_SIZE = 8
GRAD_ACCUM = 2  # accumulate this many micro-batches per optimizer update
LR = 3e-4
WEIGHT_DECAY = 0.01
ETA_MIN = 1e-5

EVAL_EVERY = 500
EVAL_BATCHES = 50
LOG_EVERY = 10  # log train loss every N steps
SAVE_EVERY = 500

NUM_WORKERS = 2
PIN_MEMORY = True
PERSISTENT_WORKERS = True
PREFETCH_FACTOR = 4

# Best-effort de-correlation: i.i.d-ish sampling with replacement
USE_REPLACEMENT_SAMPLER = True

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
USE_AMP = True and (DEVICE == 'cuda')

OUT_DIR = Path('/kaggle/working/plain_transformer_run')
CKPT_PATH = OUT_DIR / 'ckpt.pt'
RESUME_PATH = None  # set to CKPT_PATH to resume, or leave None to start fresh

OUT_DIR.mkdir(parents=True, exist_ok=True)
print('DEVICE:', DEVICE, 'AMP:', USE_AMP)
print('OUT_DIR:', OUT_DIR)


In [ ]:
# -------------------------
# Self-contained code (write a tiny module file)
# -------------------------

MODULE_PATH = Path('musicgen_kaggle_min.py')

MODULE_CODE = r'''
from __future__ import annotations

import json
import os
from dataclasses import dataclass
from pathlib import Path
from typing import Callable, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset


PathLike = str | os.PathLike[str]


@dataclass(frozen=True)
class MemmapSplitMeta:
    split: str
    num_examples: int
    tokens_per_example: int
    bars_per_sample: int
    num_attributes: int
    pad_id: int
    bar_id: int


class MemmapMusicDataset(Dataset):
    def __init__(self, split_dir: PathLike):
        self.split_dir = Path(split_dir)
        payload = json.loads((self.split_dir / 'meta.json').read_text(encoding='utf-8'))

        self.meta = MemmapSplitMeta(
            split=str(payload.get('split', self.split_dir.name)),
            num_examples=int(payload['num_examples']),
            tokens_per_example=int(payload.get('tokens_per_example', payload.get('block_size', 1024))),
            bars_per_sample=int(payload.get('bars_per_sample', 8)),
            num_attributes=4,
            pad_id=int(payload['token_ids']['pad_id']),
            bar_id=int(payload['token_ids']['bar_id']),
        )

        self.x_path = self.split_dir / 'X.u16.memmap'
        self.bi_path = self.split_dir / 'bar_indices.u8.memmap'
        self.a_path = self.split_dir / 'attributes.u8.memmap'

        self._pid: Optional[int] = None
        self._x = None
        self._bi = None
        self._a = None

    def __len__(self) -> int:
        return self.meta.num_examples

    def _ensure_open(self) -> None:
        pid = os.getpid()
        if self._pid == pid and self._x is not None:
            return

        n = self.meta.num_examples
        t = self.meta.tokens_per_example
        b = self.meta.bars_per_sample
        a = self.meta.num_attributes

        self._x = np.memmap(self.x_path, mode='r', dtype=np.uint16, shape=(n, t))
        self._bi = np.memmap(self.bi_path, mode='r', dtype=np.uint8, shape=(n, t))
        self._a = np.memmap(self.a_path, mode='r', dtype=np.uint8, shape=(n, b, a))
        self._pid = pid

    def __getitem__(self, idx: int):
        self._ensure_open()
        return self._x[idx], self._bi[idx], self._a[idx]


def make_collate_fn(*, pad_id: int) -> Callable:
    def collate(batch):
        xs, bis, attrs = zip(*batch)
        x_np = np.stack(xs, axis=0)
        bi_np = np.stack(bis, axis=0)
        a_np = np.stack(attrs, axis=0)

        X = torch.from_numpy(x_np.astype(np.int64, copy=False))
        bar_indices = torch.from_numpy(bi_np.astype(np.int64, copy=False))
        attributes = torch.from_numpy(a_np.astype(np.int64, copy=False))

        Y = torch.empty_like(X)
        Y[:, :-1] = X[:, 1:]
        Y[:, -1] = int(pad_id)

        return {'X': X, 'Y': Y, 'bar_indices': bar_indices, 'attributes': attributes}

    return collate


@dataclass(frozen=True)
class PlainTransformerConfig:
    vocab_size: int = 195
    block_size: int = 1024
    n_layers: int = 6
    n_embed: int = 512
    n_head: int = 8
    d_ff: int = 2048
    bars_per_sample: int = 8
    dropout: float = 0.1

    num_attributes: int = 4
    attribute_bins: int = 8
    attribute_embed_dim: int = 32


class AttributeConditioner(nn.Module):
    def __init__(self, cfg: PlainTransformerConfig):
        super().__init__()
        self.embeds = nn.ModuleList(
            [nn.Embedding(cfg.attribute_bins, cfg.attribute_embed_dim) for _ in range(cfg.num_attributes)]
        )
        self.proj = nn.Linear(cfg.attribute_embed_dim * cfg.num_attributes, cfg.n_embed)

    def forward(self, attributes: torch.Tensor) -> torch.Tensor:
        a0 = self.embeds[0](attributes[..., 0])
        a1 = self.embeds[1](attributes[..., 1])
        a2 = self.embeds[2](attributes[..., 2])
        a3 = self.embeds[3](attributes[..., 3])
        A = torch.cat([a0, a1, a2, a3], dim=-1)
        return self.proj(A)


class CausalSelfAttention(nn.Module):
    def __init__(self, cfg: PlainTransformerConfig):
        super().__init__()
        assert cfg.n_embed % cfg.n_head == 0
        self.n_head = cfg.n_head
        self.head_dim = cfg.n_embed // cfg.n_head
        self.qkv = nn.Linear(cfg.n_embed, 3 * cfg.n_embed, bias=True)
        self.proj = nn.Linear(cfg.n_embed, cfg.n_embed, bias=True)
        self.dropout = float(cfg.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.split(C, dim=-1)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        y = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=None,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=True,
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y)


class FiLMBlock(nn.Module):
    def __init__(self, cfg: PlainTransformerConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.n_embed)
        self.attn = CausalSelfAttention(cfg)
        self.resid_drop = nn.Dropout(cfg.dropout)

        self.film = nn.Linear(cfg.n_embed, 2 * cfg.n_embed, bias=True)

        self.ln2 = nn.LayerNorm(cfg.n_embed)
        self.mlp = nn.Sequential(
            nn.Linear(cfg.n_embed, cfg.d_ff),
            nn.GELU(),
            nn.Linear(cfg.d_ff, cfg.n_embed),
        )
        self.mlp_drop = nn.Dropout(cfg.dropout)

    def forward(self, x: torch.Tensor, cond_tok: torch.Tensor) -> torch.Tensor:
        h = self.ln1(x)
        gamma, beta = self.film(cond_tok).chunk(2, dim=-1)
        h = h * (1.0 + gamma) + beta
        x = x + self.resid_drop(self.attn(h))
        x = x + self.mlp_drop(self.mlp(self.ln2(x)))
        return x


class PlainTransformerDecoder(nn.Module):
    def __init__(self, cfg: PlainTransformerConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.n_embed)
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.n_embed)
        self.attr = AttributeConditioner(cfg)
        self.blocks = nn.ModuleList([FiLMBlock(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.n_embed)
        self.head = nn.Linear(cfg.n_embed, cfg.vocab_size, bias=True)

    def forward(self, X, bar_indices, attributes, *, targets=None, pad_id=None):
        B, T = X.shape
        if T > self.cfg.block_size:
            raise ValueError('T > block_size')

        pos = torch.arange(T, device=X.device)
        x = self.tok_emb(X) + self.pos_emb(pos)[None, :, :]

        cond_bars = self.attr(attributes)  # [B, bars, C]
        idx = bar_indices.to(dtype=torch.long).clamp_(0, self.cfg.bars_per_sample - 1)
        cond_tok = cond_bars.gather(1, idx.unsqueeze(-1).expand(-1, -1, self.cfg.n_embed))

        for blk in self.blocks:
            x = blk(x, cond_tok)

        x = self.ln_f(x)
        logits = self.head(x)

        out = {'logits': logits}
        if targets is not None:
            if pad_id is None:
                raise ValueError('pad_id required when targets provided')
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
                ignore_index=int(pad_id),
                reduction='mean',
            )
            n_tokens = int((targets != int(pad_id)).sum().item())
            out.update({'loss': loss, 'n_tokens': n_tokens})
        return out
'''

MODULE_PATH.write_text(MODULE_CODE, encoding='utf-8')
print('Wrote module:', MODULE_PATH)

# Import from the written file
import importlib
import musicgen_kaggle_min as mg
print('Imported self-contained module:', mg.__file__)


In [ ]:
# -------------------------
# Locate dataset splits
# -------------------------

def require_dir(p: Path) -> Path:
    if not p.exists() or not p.is_dir():
        raise FileNotFoundError(f"missing directory: {p}")
    return p

def require_file(p: Path) -> Path:
    if not p.exists() or not p.is_file():
        raise FileNotFoundError(f"missing file: {p}")
    return p

BASE = require_dir(Path('/kaggle/input/datasets') / KAGGLE_USER)

TRAIN_ROOT = require_dir(BASE / TRAIN_DATASET)
VAL_ROOT = require_dir(BASE / VAL_DATASET)

TRAIN_DIR = require_dir(TRAIN_ROOT / SPLIT_ROOT_REL)
VAL_DIR = require_dir(VAL_ROOT / SPLIT_ROOT_REL)

for d in [TRAIN_DIR, VAL_DIR]:
    require_file(d / 'meta.json')
    require_file(d / 'X.u16.memmap')
    require_file(d / 'bar_indices.u8.memmap')
    require_file(d / 'attributes.u8.memmap')

print('TRAIN_DIR:', TRAIN_DIR)
print('VAL_DIR:', VAL_DIR)


In [ ]:
# -------------------------
# DataLoader (best-effort de-correlation)
# -------------------------

from torch.utils.data import DataLoader, RandomSampler

train_ds = mg.MemmapMusicDataset(TRAIN_DIR)
val_ds = mg.MemmapMusicDataset(VAL_DIR)

print('train meta:', train_ds.meta)
print('val meta:', val_ds.meta)

# Collate function derives Y=shift(X)+pad and casts to torch.long
collate_fn = mg.make_collate_fn(pad_id=train_ds.meta.pad_id)

if USE_REPLACEMENT_SAMPLER:
    # i.i.d-ish sampling in step-based training
    # Need enough samples for MAX_STEPS optimizer updates, each with GRAD_ACCUM micro-batches.
    sampler = RandomSampler(train_ds, replacement=True, num_samples=BATCH_SIZE * MAX_STEPS * GRAD_ACCUM)
    shuffle = False
else:
    sampler = None
    shuffle = True

train_dl = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    shuffle=shuffle,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSISTENT_WORKERS if NUM_WORKERS > 0 else False,
    prefetch_factor=PREFETCH_FACTOR if NUM_WORKERS > 0 else None,
    drop_last=True,
    collate_fn=collate_fn,
)

val_dl = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
    drop_last=False,
    collate_fn=collate_fn,
)

# Quick batch sanity
b = next(iter(train_dl))
print({k: tuple(v.shape) for k, v in b.items()})


In [ ]:
# -------------------------
# Model + optimizer + cosine schedule (10K steps)
# -------------------------

# Spec vocab size is 195 (REMI vocab). pad_id comes from split meta.
vocab_size = 195
pad_id = int(train_ds.meta.pad_id)

cfg = mg.PlainTransformerConfig(
    vocab_size=vocab_size,
    block_size=int(train_ds.meta.tokens_per_example),
    bars_per_sample=int(train_ds.meta.bars_per_sample),
)
model = mg.PlainTransformerDecoder(cfg).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print('params:', f'{n_params/1e6:.2f}M')

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# Cosine annealing over exactly MAX_STEPS optimizer updates
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=MAX_STEPS, eta_min=ETA_MIN)

scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)


In [ ]:
# -------------------------
# Checkpointing (save/resume)
# -------------------------

def save_ckpt(step: int):
    payload = {
        'step': step,
        'model': model.state_dict(),
        'opt': opt.state_dict(),
        'scheduler': scheduler.state_dict(),
        'scaler': scaler.state_dict() if USE_AMP else None,
        'cfg': cfg.__dict__,
    }
    torch.save(payload, CKPT_PATH)


def try_resume(path: Path | None) -> int:
    if path is None:
        return 0
    path = Path(path)
    if not path.exists():
        print('No checkpoint found at', path, '(starting fresh)')
        return 0

    ckpt = torch.load(path, map_location='cpu')
    model.load_state_dict(ckpt['model'])
    opt.load_state_dict(ckpt['opt'])
    if 'scheduler' in ckpt and ckpt['scheduler'] is not None:
        scheduler.load_state_dict(ckpt['scheduler'])
    if USE_AMP and ckpt.get('scaler') is not None:
        scaler.load_state_dict(ckpt['scaler'])

    start_step = int(ckpt.get('step', 0))
    print('Resumed from step', start_step)
    return start_step

start_step = try_resume(RESUME_PATH)


In [ ]:
# -------------------------
# Train / Eval loops
# -------------------------

import time
from tqdm.auto import tqdm

@torch.no_grad()
def eval_val_loss(max_batches: int = EVAL_BATCHES) -> float:
    model.eval()
    losses = []
    for i, batch in enumerate(val_dl):
        if i >= max_batches:
            break
        X = batch['X'].to(DEVICE, non_blocking=True)
        Y = batch['Y'].to(DEVICE, non_blocking=True)
        bar_indices = batch['bar_indices'].to(DEVICE, non_blocking=True)
        attributes = batch['attributes'].to(DEVICE, non_blocking=True)
        out = model(X, bar_indices, attributes, targets=Y, pad_id=pad_id)
        losses.append(float(out['loss'].item()))
    model.train()
    return float(sum(losses) / max(len(losses), 1))

def fmt_hms(seconds: float) -> str:
    seconds = int(max(0, seconds))
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60
    return f"{h}:{m:02d}:{s:02d}"

def train():
    model.train()

    it = iter(train_dl)
    ema = None

    # For speed/ETA estimates (account for grad accumulation)
    tokens_per_batch = BATCH_SIZE * int(train_ds.meta.tokens_per_example) * GRAD_ACCUM
    last_t = time.perf_counter()
    last_step = start_step

    pbar = tqdm(range(start_step + 1, MAX_STEPS + 1), total=MAX_STEPS - start_step, desc='Training')
    for step in pbar:
        try:
            batch = next(it)
        except StopIteration:
            it = iter(train_dl)
            batch = next(it)

        X = batch['X'].to(DEVICE, non_blocking=True)
        Y = batch['Y'].to(DEVICE, non_blocking=True)
        bar_indices = batch['bar_indices'].to(DEVICE, non_blocking=True)
        attributes = batch['attributes'].to(DEVICE, non_blocking=True)

        # Gradient accumulation: do GRAD_ACCUM micro-batches per optimizer update
        opt.zero_grad(set_to_none=True)

        # Use non-deprecated AMP autocast
        ctx = (
            torch.amp.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_AMP)
            if DEVICE == 'cuda'
            else torch.autocast('cpu', enabled=False)
        )

        loss_sum = 0.0
        n_tokens_sum = 0

        for micro in range(GRAD_ACCUM):
            # fetch next micro-batch
            if micro > 0:
                try:
                    batch = next(it)
                except StopIteration:
                    it = iter(train_dl)
                    batch = next(it)

                X = batch['X'].to(DEVICE, non_blocking=True)
                Y = batch['Y'].to(DEVICE, non_blocking=True)
                bar_indices = batch['bar_indices'].to(DEVICE, non_blocking=True)
                attributes = batch['attributes'].to(DEVICE, non_blocking=True)

            with ctx:
                out = model(X, bar_indices, attributes, targets=Y, pad_id=pad_id)
                loss = out['loss'] / GRAD_ACCUM

            scaler.scale(loss).backward()

            loss_sum += float(out['loss'].item())
            n_tokens_sum += int(out.get('n_tokens', 0))

        scaler.step(opt)
        scaler.update()
        scheduler.step()  # 1 step per optimizer update

        loss_f = float(loss_sum / GRAD_ACCUM)
        ema = loss_f if ema is None else (0.95 * ema + 0.05 * loss_f)
        lr_now = float(opt.param_groups[0]['lr'])

        # Update tqdm UI + write a log line every LOG_EVERY steps
        if step == start_step + 1 or step % LOG_EVERY == 0:
            now = time.perf_counter()
            dt = now - last_t
            steps_done = step - last_step
            it_s = steps_done / max(dt, 1e-9)
            tok_s = (steps_done * tokens_per_batch) / max(dt, 1e-9)
            eta_s = (MAX_STEPS - step) / max(it_s, 1e-9)

            msg = (
                f"step {step}/{MAX_STEPS} | loss {loss_f:.4f} | ema {ema:.4f} | lr {lr_now:.2e} "
                f"| tok/s {tok_s/1e6:.2f}M | eta {fmt_hms(eta_s)}"
            )
            pbar.set_description(msg)
            tqdm.write(msg)

            last_t = now
            last_step = step

        if step % EVAL_EVERY == 0:
            v = eval_val_loss()
            tqdm.write(f"eval @ step {step:5d} | val_loss {v:.4f}")

        if step % SAVE_EVERY == 0:
            save_ckpt(step)

    save_ckpt(MAX_STEPS)
    print('Training complete. Final checkpoint:', CKPT_PATH)


train()
